# GIAI ĐOẠN 1: TIỀN XỬ LÝ VÀ LÀM SẠCH DỮ LIỆU (DATA CLEANING)

**Đề tài:** Phân tích rủi ro rời bỏ dịch vụ thẻ tín dụng (Customer Churn Analysis)
**Mục tiêu của Notebook:**
1. Khám phá hiện trạng, kiểm định chất lượng tệp dữ liệu thô.
2. Thực thi các quyết định làm sạch trên bản sao độc lập nhằm bảo toàn 100% dữ liệu gốc.
3. Rà soát logic miền giá trị, định lượng ngoại lai và chuẩn hóa các biến đặc trưng.
4. Lập bảng đối chiếu chứng minh chất lượng dữ liệu được cải thiện trước khi đưa vào phân tích chuyên sâu.

In [1]:
import pandas as pd
import numpy as np 

In [2]:
df_raw=pd.read_csv('BankChurners.csv')
df_clean=df_raw.copy()

## 1. Khám phá Tổng quan Dữ liệu (Data Audit)
Bước đầu tiên là kiểm tra kích thước tập dữ liệu, rà soát các dòng trùng lặp và phát hiện các giá trị khuyết thiếu. Đặc biệt, ta cần quét các cột dạng chuỗi (Object) để tìm các giá trị khuyết bị ẩn dưới dạng nhãn văn bản (ví dụ: 'Unknown').

In [3]:
display(df_raw.head())
print(f"Dữ liệu gồm: {df_raw.shape[0]} dòng, {df_raw.shape[1]} cột")
print(f"Số dữ liệu trùng lặp: {df_raw.duplicated().sum()} dòng")
print(f"Số ô NaN (null) theo kỹ thuật: {df_raw.isna().sum().sum()}")
print(f"Số ID khách hàng bị trùng: {df_raw['CLIENTNUM'].duplicated().sum()}")

,CLIENTNUM,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,...,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2
0,768805383,Existing Customer,45,M,3,High School,Married,$60K - $80K,Blue,39,...,12691.0,777,11914.0,1.335,1144,42,1.625,0.061,0.000093,0.99991
1,818770008,Existing Customer,49,F,5,Graduate,Single,Less than $40K,Blue,44,...,8256.0,864,7392.0,1.541,1291,33,3.714,0.105,0.000057,0.99994
2,713982108,Existing Customer,51,M,3,Graduate,Married,$80K - $120K,Blue,36,...,3418.0,0,3418.0,2.594,1887,20,2.333,0.000,0.000021,0.99998
3,769911858,Existing Customer,40,F,4,High School,Unknown,Less than $40K,Blue,34,...,3313.0,2517,796.0,1.405,1171,20,2.333,0.760,0.000134,0.99987
4,709106358,Existing Customer,40,M,3,Uneducated,Married,$60K - $80K,Blue,21,...,4716.0,0,4716.0,2.175,816,28,2.500,0.000,0.000022,0.99998


Dữ liệu gồm: 10127 dòng, 23 cột
Số dữ liệu trùng lặp: 0 dòng
Số ô NaN (null) theo kỹ thuật: 0
Số ID khách hàng bị trùng: 0


In [4]:
#Đếm ô unknown
name_col=df_raw.columns
for i in name_col:
    dem_unk=(df_raw[i]=='Unknown').sum()
    if dem_unk>0:
        print(f"Cột {i} có: {dem_unk} ô Unknown ({dem_unk/len(df_raw):.2%})")
#Đếm số khách hàng chứa ít nhất 1 thông tin Unkown
so_data_unk=(df_raw[name_col]=='Unknown').any(axis=1).sum()
print(f"Số khách thiếu thông tin: {so_data_unk}/{len(df_raw)} ({so_data_unk/len(df_raw):.2%})")



Cột Education_Level có: 1519 ô Unknown (15.00%)
Cột Marital_Status có: 749 ô Unknown (7.40%)
Cột Income_Category có: 1112 ô Unknown (10.98%)
Số khách thiếu thông tin: 3046/10127 (30.08%)


## 2. Các Quyết định Xử lý Dữ liệu & Biện giải Nghiệp vụ
Dựa trên kết quả khám phá, các thao tác làm sạch sau được thực thi:
1. **Triệt tiêu nguy cơ rò rỉ dữ liệu (Data Leakage):** Loại bỏ hoàn toàn 2 cột `Naive_Bayes_Classifier_...` ở cuối bảng. Đây là xác suất dự đoán từ một mô hình bên ngoài, việc giữ lại sẽ làm hỏng tính khách quan của bài toán.
2. **Chuẩn hóa biến mục tiêu (`Attrition_Flag`):** Mã hóa nhị phân (`Existing Customer` -> 0, `Attrited Customer` -> 1) và ép kiểu `int8` để tối ưu bộ nhớ RAM, sẵn sàng cho tính toán ma trận tương quan.
3. **Chiến lược giữ nguyên nhãn 'Unknown':** Với hơn 30% mẫu quan sát chứa nhãn 'Unknown', việc xóa dòng làm hơn 30% quan sát và có thể gây lệch mẫu. Ta quyết định giữ nguyên 'Unknown' thành một nhóm danh mục độc lập.

In [5]:
#1: Xóa 2 cột gây rò rỉ dữ liệu Naive_Bayes
leak_col=[c for c in df_clean.columns if c.startswith('Naive_Bayes')]
df_clean=df_clean.drop(columns=leak_col)
#2: Chuyển biến mục tiêu Attrition_Flag thành 0/1
df_clean['Attrition_Flag']=df_clean['Attrition_Flag'].map({
    'Existing Customer':0,
    'Attrited Customer':1
}).astype('int8')

## 3. Kiểm định Logic miền giá trị và Đánh giá Ngoại lai (Outliers)
Hệ thống sẽ tự động quét toàn bộ các biến định lượng để:
- Đảm bảo không có vi phạm logic vật lý (ví dụ: tuổi âm, số tiền âm, lỗi chia cho 0).
- Sử dụng quy tắc hàng rào Tứ phân vị ($1.5 \times \text{IQR}$) và hệ số Skewness để định lượng các điểm bất thường.

### 3.1 Kiểm định Logic nghiệp vụ chéo giữa các cột

In [6]:
print("--- KIỂM ĐỊNH LOGIC NGHIỆP VỤ CHÉO GIỮA CÁC CỘT ---")

# Logic 1: Credit_Limit phải bằng Total_Revolving_Bal + Avg_Open_To_Buy
chenh_lech_han_muc = (df_clean["Credit_Limit"] - df_clean["Total_Revolving_Bal"] - df_clean["Avg_Open_To_Buy"]).abs()
print(f"1. Sai số lớn nhất của công thức Hạn mức: {chenh_lech_han_muc.max():.6f}")
print(f"   Số khách hàng có dư nợ vượt hạn mức: {(df_clean['Total_Revolving_Bal'] > df_clean['Credit_Limit']).sum()} dòng")

# Logic 2: Avg_Utilization_Ratio phải khớp với (Total_Revolving_Bal / Credit_Limit)
chenh_lech_ty_le = (df_clean["Avg_Utilization_Ratio"] - (df_clean["Total_Revolving_Bal"] / df_clean["Credit_Limit"])).abs()
print(f"2. Sai số làm tròn lớn nhất của Tỷ lệ sử dụng thẻ: {chenh_lech_ty_le.max():.4f}")

# Logic 3: Tuổi tại thời điểm bắt đầu mở thẻ phải từ 18 tuổi trở lên
tuoi_luc_mo_the = df_clean["Customer_Age"] - (df_clean["Months_on_book"] / 12)
print(f"3. Độ tuổi nhỏ nhất tại thời điểm mở thẻ: {tuoi_luc_mo_the.min():.1f} tuổi")
print(f"   Số khách hàng mở thẻ khi chưa đủ 18 tuổi: {(tuoi_luc_mo_the < 18).sum()} dòng")

--- KIỂM ĐỊNH LOGIC NGHIỆP VỤ CHÉO GIỮA CÁC CỘT ---
1. Sai số lớn nhất của công thức Hạn mức: 0.000000
   Số khách hàng có dư nợ vượt hạn mức: 0 dòng
2. Sai số làm tròn lớn nhất của Tỷ lệ sử dụng thẻ: 0.0005
3. Độ tuổi nhỏ nhất tại thời điểm mở thẻ: 23.0 tuổi
   Số khách hàng mở thẻ khi chưa đủ 18 tuổi: 0 dòng


### 3.2 Kiểm định tính hợp lệ và ngoại lai

In [7]:
cols=[c for c in (df_clean.select_dtypes(include=['number']).columns)
         if c != 'Attrition_Flag' and c!= 'CLIENTNUM']
tao_bang=[]
for col in cols:
    s=df_clean[col]
    #1 Kiểm tra logic cận biên
    check_neg=int((s<0).sum())
    check_infi=int((~np.isfinite(s)).sum())
    #2 Kiểm tra các giá trị ngoại lai
    q1=s.quantile(0.25)
    q3=s.quantile(0.75)
    iqr=q3-q1
    lower_bound=q1-1.5*iqr
    upper_bound=q3+1.5*iqr
    num_outlier = int(((s < lower_bound) | (s > upper_bound)).sum())
    pr_outlier = f"{num_outlier / len(df_clean):.2%}"

    tao_bang.append({
    'Thuộc tính':col,
    'Min':round(s.min(),2),
    'Max':round(s.max(),2),
    'Số giá trị âm':check_neg,
    'Số giá trị vô cùng':check_infi,
    'Độ lệch (Skew)':round(s.skew(),2),
    'Số giá trị ngoại lai':num_outlier,
    'Tỉ lệ ngoại lai':pr_outlier
    })
# Xuất bảng tổng hợp kết quả
df_report = pd.DataFrame(tao_bang)
print("--- BẢNG KIỂM ĐỊNH TÍNH HỢP LỆ VÀ NGOẠI LAI ---")
display(df_report.sort_values(by="Số giá trị ngoại lai", ascending=False).reset_index(drop=True))

--- BẢNG KIỂM ĐỊNH TÍNH HỢP LỆ VÀ NGOẠI LAI ---


,Thuộc tính,Min,Max,Số giá trị âm,Số giá trị vô cùng,Độ lệch (Skew),Số giá trị ngoại lai,Tỉ lệ ngoại lai
0,Credit_Limit,1438.3,34516.00,0,0,1.67,984,9.72%
1,Avg_Open_To_Buy,3.0,34516.00,0,0,1.66,963,9.51%
2,Total_Trans_Amt,510.0,18484.00,0,0,2.04,896,8.85%
3,Contacts_Count_12_mon,0.0,6.00,0,0,0.01,629,6.21%
4,Total_Amt_Chng_Q4_Q1,0.0,3.40,0,0,1.73,396,3.91%
5,Total_Ct_Chng_Q4_Q1,0.0,3.71,0,0,2.06,394,3.89%
6,Months_on_book,13.0,56.00,0,0,-0.11,386,3.81%
7,Months_Inactive_12_mon,0.0,6.00,0,0,0.63,331,3.27%
8,Customer_Age,26.0,73.00,0,0,-0.03,2,0.02%
9,Total_Trans_Ct,10.0,139.00,0,0,0.15,2,0.02%


**Kết luận kiểm tra Logic & Ngoại lai (Outliers):**
1. **Dữ liệu hợp lý 100%:**
   - Không có cột nào bị âm hoặc lỗi chia cho 0. Các con số đều thực tế (Tuổi từ 26–73, Tỷ lệ dùng thẻ từ 0–1, Số tháng ngừng giao dịch từ 0–6 tháng).
   - Các cột khớp công thức với nhau: Không ai nợ vượt hạn mức (`Credit_Limit = Total_Revolving_Bal + Avg_Open_To_Buy`), và độ tuổi mở thẻ nhỏ nhất là 23 tuổi.
2. **Về giá trị ngoại lai (Outliers):**
   - Các cột như Tuổi (`Customer_Age`) và Số lần quẹt (`Total_Trans_Ct`) phân bố đều, hầu như không có ngoại lai.
   - Các cột tiền bạc (`Credit_Limit`, `Total_Trans_Amt`) lệch phải mạnh và có khoảng 8,8% – 9,7% điểm vượt ngưỡng IQR.
3. **Quyết định xử lý:** **Giữ nguyên 100% các điểm ngoại lai này** vì đây là nhóm khách hàng VIP có hạn mức và mức chi tiêu cao thực tế của ngân hàng, không phải lỗi nhập liệu.

## 4. Bằng chứng Định lượng: Đối chiếu Trước và Sau (Before vs. After)
So sánh bộ dữ liệu gốc (`df_raw`) và bộ dữ liệu đã làm sạch (`df_clean`) để chứng minh tính hiệu quả của quy trình tiền xử lý.

In [8]:
comparison_df = pd.DataFrame({
    "Tiêu chí": [
        "Số dòng (Rows)", 
        "Tổng số cột", 
        "Số dòng trùng lặp", 
        "Số cột rò rỉ đã loại bỏ (Naive Bayes)", 
        "Kiểu dữ liệu biến mục tiêu", 
        "Bộ nhớ RAM (MB)"
    ],
    "df_raw (Trước)": [
        len(df_raw), 
        df_raw.shape[1], 
        df_raw.duplicated().sum(), 
        0, 
        str(df_raw["Attrition_Flag"].dtype), 
        round(df_raw.memory_usage(deep=True).sum() / 1024**2, 2)
    ],
    "df_clean (Sau)": [
        len(df_clean), 
        df_clean.shape[1], 
        df_clean.duplicated().sum(), 
        len(leak_col), 
        str(df_clean["Attrition_Flag"].dtype), 
        round(df_clean.memory_usage(deep=True).sum() / 1024**2, 2)
    ]
})

print("\n--- BẢNG ĐỐI CHIẾU BEFORE vs AFTER ---")
print(comparison_df.to_string(index=False))

#Xuất dữ liệu
df_clean.to_csv("BankChurners_clean.csv",index=False)
print("\n-> Đã xuất tệp 'BankChurners_clean.csv' thành công.")



--- BẢNG ĐỐI CHIẾU BEFORE vs AFTER ---
                             Tiêu chí df_raw (Trước) df_clean (Sau)
                       Số dòng (Rows)          10127          10127
                          Tổng số cột             23             21
                    Số dòng trùng lặp              0              0
Số cột rò rỉ đã loại bỏ (Naive Bayes)              0              2
           Kiểu dữ liệu biến mục tiêu            str           int8
                      Bộ nhớ RAM (MB)           2.25           1.87

-> Đã xuất tệp 'BankChurners_clean.csv' thành công.
